In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

In [2]:
data_idx = 29

In [3]:
model_path = "/mnt/petrelfs/share_data/huzican/Qwen-2.5-Math-7B-SimpleRL-Zoo"

tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,  # 使用全部8张GPU
    gpu_memory_utilization=0.85,  # 可以设置更高的内存利用率
    dtype="auto"
)

INFO 08-27 18:35:36 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 08-27 18:35:36 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/share_data/huzican/Qwen-2.5-Math-7B-SimpleRL-Zoo', speculative_config=None, tokenizer='/mnt/petrelfs/share_data/huzican/Qwen-2.5-Math-7B-SimpleRL-Zoo', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), seed=0, served_model_name=/mnt/petrelfs/share_data/huzican/Qwen-2.5-Math-7B-SimpleRL-Zoo, us

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 08-27 18:35:48 model_runner.py:732] Loading model weights took 14.2418 GB
INFO 08-27 18:35:49 gpu_executor.py:102] # GPU blocks: 60277, # CPU blocks: 4681
INFO 08-27 18:35:52 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 08-27 18:35:52 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 08-27 18:36:06 model_runner.py:1225] Graph capturing finished in 14 secs.


In [4]:
train_data_path = "dataset/openr1.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)
test_data_path = "dataset/aime.parquet"
test_dataset = RLHFDataset(parquet_files=test_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

test_dataloader = DataLoader(dataset=test_dataset,
                            batch_size=1,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 45792
filter dataset len: 45764
original dataset len: 30
filter dataset len: 30


In [5]:
test_data = train_dataset[data_idx]
# print(test_data['reward_model'])
input_text = tokenizer.decode(test_data['input_ids'], skip_special_tokens=True)

In [6]:
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=1.0,
    max_tokens=8192,
)

prompts = [input_text] * 8
outputs = base_model.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 8/8 [00:13<00:00,  1.65s/it, est. speed input: 116.11 toks/s, output: 489.17 toks/s]


In [7]:
group_rollout = []
for output in outputs:
    # 提取生成的文本
    full_text = output.outputs[0].text
    # 只保留输入之后新生成的部分
    generated_text = full_text[len(input_text):]
    group_rollout.append(generated_text)

In [8]:
print(test_data['reward_model'])
for i in range(len(group_rollout)):
    print(f"********{i}**********")
    print(group_rollout[i])  # 从列表中提取文本
    print("__________end_____________\n")

{'ground_truth': '\\sqrt{3}', 'style': 'rule'}
********0**********
angled triangles, each with legs of length \(a\).
   - The area of one of these triangles is \(\frac{1}{2}a^2\).
   - So, the total lateral surface area \(A_{\text{lateral}}\) is \(3 \times \frac{1}{2}a^2 = \frac{3}{2}a^2\).

4. **Find the Ratio:**
   - The ratio of the lateral surface area to the area of the base is \(\frac{A_{\text{lateral}}}{A_{\text{base}}} = \frac{\frac{3}{2}a^2}{\frac{\sqrt{3}}{2}a^2} = \frac{3}{\sqrt{3}} = \sqrt{3}\).

So, the ratio of the lateral surface area of the pyramid to the area of its base is \(\sqrt{3}\).

Let's confirm this with Python and sympy:

```python
import sympy as sp

# Define the side length of the equilateral triangle base
a = sp.symbols('a')

# Area of the base (equilateral triangle with side length a*sqrt(2))
A_base = (sp.sqrt(3) / 2) * a**2

# Area of one lateral face (right-angled triangle with legs of length a)
A_lateral_face = (1 / 2) * a**2

# Total lateral surface ar